# Расчёт метрик устойчивости сегментации

Выполните ячейки по порядку из корня репозитория. Поддерживаются **GrabCut, Berkeley, DAVIS и COCO-MVal**.

Для каждого файла весов ожидаются папки `имя_модели_MIN/plots` и `имя_модели_MAX/plots` с результатами запуска `--iou-analysis`. Базовые результаты можно положить в `имя_модели/plots`. Если их нет, метрики минимума и максимума всё равно рассчитываются.


In [ ]:
from pathlib import Path
import pickle
import numpy as np
import yaml

from isegm.inference.datasets import SUPPORTED_DATASETS, normalize_dataset_name


## Папка результатов

По умолчанию используется путь из `config.yml` и стандартные подпапки журналов. Если вы задали `--logs-path` или объединили результаты в другой папке, измените `exp_path` вручную. В ней должны лежать папки моделей, а не непосредственно файлы метрик.


In [ ]:
with open('config.yml', encoding='utf-8') as stream:
    config = yaml.safe_load(stream)
exp_path = Path(config['EXPS_PATH']) / 'evaluation_logs' / 'others'


## Загрузка траекторий

Читаются только результаты четырёх поддерживаемых датасетов. Прежнее имя `COCO_MVal` преобразуется в `COCO-MVal`. Для расчёта нужны файлы `.pickle` из подпапок `plots`, содержащие траектории IoU и BIoU.


In [ ]:
data_dict = {}

if not exp_path.is_dir():
    raise FileNotFoundError(f'Папка результатов не найдена: {exp_path}. Проверьте exp_path и выполните оценку моделей.')

for model_dir in sorted(exp_path.iterdir()):
    plots_dir = model_dir / 'plots'
    if not plots_dir.is_dir():
        continue

    model_name = model_dir.name
    suffix = 'STANDARD'
    if model_name.endswith(('_MIN', '_MAX')):
        suffix = model_name[-3:]
        model_name = model_name[:-4]

    for results_path in sorted(plots_dir.glob('*.pickle')):
        with results_path.open('rb') as stream:
            data = pickle.load(stream)

        # Имя из метаданных сохраняет COCO-MVal целиком, включая суффикс MVal.
        dataset_name = data.get('dataset_name')
        if dataset_name is None:
            dataset_name = next((name for name in (*SUPPORTED_DATASETS, 'COCO_MVal')
                                 if results_path.name.startswith(name + '_')), '')
        try:
            dataset_name = normalize_dataset_name(dataset_name)
        except ValueError:
            continue

        trajectories = data['all_ious']
        if len(trajectories) == 0:
            continue
        metrics = data_dict.setdefault(model_name, {}).setdefault(dataset_name, {})
        metrics[suffix + '_IOU'] = np.array([item[1][:, 0] for item in trajectories])
        metrics[suffix + '_BIOU'] = np.array([item[1][:, 1] for item in trajectories])

inverse_index = {}
for model_name, datasets in data_dict.items():
    for dataset_name, metrics in datasets.items():
        inverse_index.setdefault(dataset_name, {})[model_name] = metrics

print('Загружены датасеты:', ', '.join(inverse_index) or 'нет результатов')
print('Загружены модели:', ', '.join(data_dict) or 'нет результатов')


## Выбор моделей и датасетов

По умолчанию выводятся все найденные модели и четыре датасета. При необходимости замените `models_to_print` списком имён нужных моделей. Для отсутствующих результатов выводится пояснение.


In [ ]:
models_to_print = sorted(data_dict)
datasets_to_print = list(SUPPORTED_DATASETS)


## Итоговые метрики

Значения в процентах — нормированная площадь под средней кривой качества по кликам. «Разница» равна максимуму минус минимум; «База» соответствует базовой стратегии кликов. Для единственной точки используется её значение.


In [ ]:
def trajectory_score(values):
    mean_curve = values.mean(axis=0)
    if len(mean_curve) == 1:
        return float(mean_curve[0])
    return float(np.trapz(mean_curve) / (len(mean_curve) - 1))


sep = '-' * 40
for dataset_name in datasets_to_print:
    print(sep)
    print(dataset_name)
    for model_name in models_to_print:
        print(sep)
        print(model_name)
        metrics = inverse_index.get(dataset_name, {}).get(model_name, {})
        for metric in ['IOU', 'BIOU']:
            if 'MAX_' + metric not in metrics or 'MIN_' + metric not in metrics:
                print(metric, '| Нет результатов минимизации или максимизации')
                continue

            maximum = trajectory_score(metrics['MAX_' + metric])
            minimum = trajectory_score(metrics['MIN_' + metric])
            baseline_values = metrics.get('STANDARD_' + metric)
            baseline = (f'{100 * trajectory_score(baseline_values):.2f}'
                        if baseline_values is not None else 'нет данных')
            print(f'{metric:<4} | Мин {100 * minimum:.2f} | База {baseline} '
                  f'| Макс {100 * maximum:.2f} | Разница {100 * (maximum - minimum):.2f}')
